In [ ]:
%pip install -q scikit-learn==1.8.0

In [ ]:
from pathlib import Path

from google.colab import files

data_dir = Path("team1_data")
data_dir.mkdir(exist_ok=True)
required = ("train.csv", "test.csv", "sample_submission.csv")
uploaded = files.upload()
for name in required:
    if name in uploaded:
        (data_dir / name).write_bytes(uploaded[name])
missing = [name for name in required if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError("Upload these files: " + ", ".join(missing))


In [ ]:
%%writefile team1_model.py

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

SEED = 410
RARE = "DMADV"
 CUES = [
    r"root cause|unknown|investigat\w*|find out|why|diagnos\w*|analy[sz]\w*",
    r"trend|baseline|metric\w*|benchmark\w*|monitor\w*|data|measur\w*",
    r"handoff|hand.off|workflow|escalat\w*|coordinat\w*|communicat\w*",
    r"workshop|meeting\w*|team\w*|collaborat\w*|stand.up\w*",
    r"recommend\w*|correct\w*|replac\w*|remov\w*|cancel\w*|fix\w*",
    r"new (?:process|system|procedure|program|method)|design\w*|develop\w*|creat\w*",
    r"lack\w*|no existing|does not exist|cannot meet|incapable",
    r"staff\w*|fund\w*|budget\w*|resource\w*|purchas\w*|polic\w*|refund\w*",
    r"fail\w*|error\w*|inconsisten\w*|delay\w*|repeat\w*|inefficien\w*",
]


def route_cues(texts):
    return csr_matrix([
        [min(3, len(re.findall(cue, text.lower()))) / 3 for cue in CUES]
        for text in texts
    ])


def text_features(include_cues=False):
    branches = [
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=2500, sublinear_tf=True)),
        ("character", TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=2500, sublinear_tf=True
        )),
    ]
    if include_cues:
        branches.append(("route_cues", FunctionTransformer(route_cues)))
    return FeatureUnion(branches)


def evaluate(model, texts, labels, folds, common_labels):
    scores = []
    for fit_rows, dev_rows in folds:
        model.fit(texts.iloc[fit_rows], labels.iloc[fit_rows])
        predicted = model.predict(texts.iloc[dev_rows])
        scores.append(f1_score(
            labels.iloc[dev_rows], predicted, labels=common_labels,
            average="macro", zero_division=0,
        ))
    return np.mean(scores)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("data_dir", type=Path)
    args = parser.parse_args()
    train = pd.read_csv(args.data_dir / "train.csv")
    texts, labels = train["text"], train["label"]
    rare_rows = np.flatnonzero(labels.eq(RARE))
    common_rows = np.flatnonzero(labels.ne(RARE))
    common_labels = sorted(labels.iloc[common_rows].unique())

     splitter = RepeatedStratifiedKFold(n_splits=4, n_repeats=5, random_state=SEED)
    folds = [
        (np.r_[common_rows[fit], rare_rows], common_rows[dev])
        for fit, dev in splitter.split(common_rows, labels.iloc[common_rows])
    ]
    candidates = {
        "linear TF-IDF": Pipeline([
            ("features", text_features()),
            ("classifier", LinearSVC(C=1, random_state=SEED)),
        ]),
        "linear TF-IDF + cues": Pipeline([
            ("features", text_features(include_cues=True)),
            ("classifier", LinearSVC(C=1, random_state=SEED)),
        ]),
        "neural TF-IDF + cues": Pipeline([
            ("features", text_features(include_cues=True)),
            ("classifier", MLPClassifier(
                hidden_layer_sizes=(24,), solver="adam", alpha=0.1, learning_rate_init=0.005,
                max_iter=400, random_state=SEED,
            )),
        ]),
    }
    baseline = DummyClassifier(strategy="most_frequent")
    score = evaluate(baseline, texts, labels, folds, common_labels)
    print(f"Local majority baseline: {score:.4f}")
    scores = {
        name: evaluate(model, texts, labels, folds, common_labels)
        for name, model in candidates.items()
    }
    for name, score in scores.items():
        print(f"{name}: {score:.4f}")
    selected = max(scores, key=scores.get)
    model = candidates[selected].fit(texts, labels)

    test = pd.read_csv(args.data_dir / "test.csv", dtype={"id": str})
    sample = pd.read_csv(args.data_dir / "sample_submission.csv", dtype={"id": str})
    assert test.id.is_unique and sample.id.is_unique
    assert set(test.id) == set(sample.id) and len(test) == 20
    predictions = pd.Series(model.predict(test["text"]), index=test["id"])
    submission = sample[["id"]].assign(label=sample.id.map(predictions))
    assert submission.label.isin(labels.unique()).all()
    submission.to_csv("team1_run1.csv", index=False)
    print(f"run1: {selected}, 4-class CV macro-F1 {scores[selected]:.3f}, seed {SEED}")


if __name__ == "__main__":
    main()


In [ ]:
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python team1_model.py team1_data

In [ ]:
files.download("team1_run1.csv")